# Forecasting Hourly Day-Ahead Electricity Prices in the German-Luxembourg Bidding Zone
**DAI Mission — Data & AI in Economics | TU Dortmund**

---

## Team

| Name | Role   |
|------|--------|
| Lennart Oberkönig | Lead   |
| Tim Janis Schmale | Member |

---

**LLM Assistance Disclosure**
  
We used Microsoft Copilot, GitHub Copilot, and ChatGPT to help with selected technical parts of our work, such as setting up the supervised learning pipeline, debugging our own code, and creating clear and visually appealing figures. We also used these tools to smooth out some transitions and improve the overall flow of our written text.
All analyses, interpretations, and conclusions are entirely our own.

## Research Question

*How accurately can hourly day-ahead electricity prices in the German-Luxembourg bidding zone be forecasted using market fundamentals, renewable generation forecasts, load forecasts and calendar effects?*

In [1]:
## Packages

# basic packages
import numpy as np
import pandas as pd
from pathlib import Path
import re
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

# plotting
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns

# sklearn models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import silhouette_score
from sklearn.linear_model import Lasso


# preprocessing and pipelines
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.base import clone

# metrics and model inspection
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# clustering and manifold learning
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

# others
import holidays
from itertools import combinations

# Causal Inference
import networkx as nx

np.random.seed(42)
print('All imports successful')

All imports successful


---
## Work Plan

| Section                      | Responsible Member              | Main Tasks                                               |
|------------------------------|---------------------------------|----------------------------------------------------------|
| §1 Research Question & Data  | Lennart Oberkönig               | Data Sourcing, Cleaning, Variable Table                  |
| §2 Causal Inference          | Tim Schmale                     | DAG Design                                               |
| §3 Supervised Learning       | Lennart Oberkönig               | Model Selection, Training, Evaluation                    |
| §4 Unsupervised / Generative | Tim Schmale                     | Method Choice, Implementation, Visualisation             |
| §5 Synthesis & Communication | Lennart Oberkönig & Tim Schmale | Cross-Method Narrative, Conclusion, Notebook Readability |

**Shared tasks:** The whole project was done together in person, but responsibilities have been determined.

---
## Section 1 — Research Question & Data

### Research Question

How accurately can hourly day-ahead electricity prices in the German-Luxembourg bidding zone be forecasted using market fundamentals, renewable generation forecasts, load forecasts and calendar effects?

### Motivation

Electricity price forecasting is a challenging and highly relevant task in modern power markets because short-term electricity prices exhibit complex dynamics and depend on the continuous balance between production and consumption, which is affected by several factors such as demand and weather conditions (Maciejowska, Uniejewski and Weron, 2022, P. 1ff.). In day-ahead electricity markets, market participants submit buy and sell orders for electricity delivery on the following day. These bids and offers are aggregated into demand and supply curves, and the market-clearing price is determined by the intersection of these curves. Thus, the hourly day-ahead price reflects the equilibrium between expected electricity demand and available supply for each delivery hour. The auction for this pricing mechanism closes each day at 12:00 and the prices for the next day are determined (Ghelasi and Ziel, 2024, P. 588f.).

Beyond its methodological relevance, electricity price forecasting also has direct economic value. Accurate day-ahead price forecasts can support market participants in planning bidding strategies, scheduling generation or consumption, managing price risk and identifying economically favorable hours for flexible assets such as storage or demand-side flexibility. In this sense, forecast accuracy is not only a statistical objective but can translate into better market decisions.

The auction-based price formation is closely related to the merit-order effect. Since electricity from renewable energy sources such as wind and solar PV is characterized by negligible marginal costs, increasing renewable feed-in tends to affect the aggregated supply curve and can reduce day-ahead electricity prices (Macedo, Marques and Damette, 2022, P. 885ff.). Therefore, renewable generation is an important explanatory factor for forecasting hourly day-ahead electricity prices. This mechanism is particularly relevant for the German-Luxembourg bidding zone, where electricity prices are closely linked to load and renewable generation. Another driving factor for the price is seasonality, since the price is showing recurring patterns on a weekly, daily and intraday level (Trebbien et al., 2024, P. 35f.). 

From a forecasting perspective, this leads to an important modeling question: whether day-ahead electricity prices should be represented as one continuous hourly time series or as a 24-dimensional daily price vector. In a univariate framework, hourly prices are treated as one high-frequency time series, and forecasts for the 24 hours of the next day are generated sequentially. This means that earlier forecasts can enter the prediction of later hours, which makes the approach sensitive to error accumulation. In contrast, the multivariate framework uses an explicit day-by-hour structure and forecasts all 24 hourly prices of the next day at once. This allows each delivery hour to have its own model structure and to capture hour-specific price patterns (Ziel and Weron, 2018, P. 397ff.). In addition, this framework can be extended by including explanatory variables such as load forecasts, wind and solar generation forecasts and calendar effects. Due to the inclusion of additional explanatory variables we choose a multivariate modeling framework for forecasting the hourly day-ahead prices.

This project combines explanatory and predictive methods to analyze hourly day-ahead electricity prices in the German-Luxembourg bidding zone: a directed acyclic graph is used to structure the assumed relationships between relevant market drivers, K-Means clustering and t-SNE are applied to explore and visualize recurring price regimes, and Decision Tree, Random Forest and Neural Network regression models are evaluated against a naive baseline to assess their forecasting performance.


### Data Sources

| Dataset                          | Source / URL                                                  | Access Method   |
|----------------------------------|---------------------------------------------------------------|-----------------|
| Forecasted Day-Ahead Generation  | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |
| Forecasted Day-Ahead Load        | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |
| Day-Ahead Electricity Price      | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |

### Dataset Information

| Variable                                                                | Type        | Role       | Description                                                                                                                                                                              |
|-------------------------------------------------------------------------|-------------|------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| timestamp                                                               | datetime    | Identifier | Start of timeperiod in Central European (Summer-) Time                                                                                                                                   |
| Total Load FC [MWh]                                                     | float       | feature    | Forecasted total electricity consumption for the following day.                                                                                                                          |
| Wind Offshore Production FC [MWh]                                       | float       | feature    | Forecasted net electricity generation from offshore wind turbines for the following day.                                                                                                 |
| Wind Onshore Production FC [MWh]                                        | float       | feature    | Forecasted net electricity generation from onshore wind turbines for the following day.                                                                                                  |
| Photovoltaik Production FC [MWh]                                        | float       | feature    | Forecasted net electricity generation from photovoltaic systems for the following day. The forecast is part of the SMARD category for forecasted wind and photovoltaic generation.       |
| Other Production FC [MWh]                                               | float       | feature    | Forecasted net electricity generation from other systems  for the following day.                                                                                                         |
| Stabilized Day Ahead Price [EUR/MWh]                                    | float       | target     | Hourly wholesale electricity price in the day-ahead market with variance stabilization approach applied.                                                                                 |
| German_holiday                                                          | boolean     | feature    | Indicator for whether the calendar date of the timestamp is a public holiday in Germany.                                                                                                 |
| Luxembourg_holiday                                                      | boolean     | feature    | Indicator for whether the calendar date of the timestamp is a public holiday in Luxembourg.                                                                                              |
| weekday                                                                 | categorical | feature    | Calendar weekday derived from the timestamp.                                                                                                                                             |
| hour                                                                    | categorical | feature    | Hour of the day derived from the timestamp.                                                                                                                                              |
| month                                                                   | categorical | feature    | Calendar month derived from the timestamp.                                                                                                                                               |
| year                                                                    | numeric     | feature    | Calendar year derived from the timestamp.                                                                                                                                                |
| Lagged Stabilized Day Ahead Price [EUR/MWh] for hour -1,...,-168        | float       | feature    | Hourly wholesale electricity price in the day-ahead market with variance stabilization approach applied for hour -1,...,-168. This entry corresponds to 168 lag features in total.       |
| Minimum of Lagged Stabilized Day Ahead Price [EUR/MWh] of day -1,...,-7 | float       | feature    | Minimum of hourly wholesale electricity price in the day-ahead market with variance stabilization approach applied for day -1,...,-7. This entry corresponds to 7 lag features in total. |
| Maximum of Lagged Stabilized Day Ahead Price [EUR/MWh] of day -1,...,-7 | float       | feature    | Maximum of hourly wholesale electricity price in the day-ahead market with variance stabilization approach applied for day -1,...,-7. This entry corresponds to 7 lag features in total. |

In [2]:
## Load data
# notebook working directory
BASE = Path().resolve()
DATA = BASE / "data"

# load the data
load_forecast = pd.read_csv(DATA / "Prognostizierter_Stromverbrauch.csv",delimiter=";")
generation_forecast = pd.read_csv(DATA / "Prognostizierte_Erzeugung_Day-Ahead.csv",delimiter=";")
day_ahead_price = pd.read_csv(DATA / "Gro_handelspreise.csv",delimiter=";")

# restrict to the desired columns, rename columns & set nans
generation_forecast = generation_forecast[[
    "Datum von",
    "Wind Offshore [MWh] Berechnete Auflösungen",	
    "Wind Onshore [MWh] Berechnete Auflösungen",	
    "Photovoltaik [MWh] Berechnete Auflösungen",	
    "Sonstige [MWh] Berechnete Auflösungen"]].rename(columns={"Datum von": "timestamp",
                                                              "Wind Offshore [MWh] Berechnete Auflösungen": "Wind Offshore Production FC [MWh]",
                                                              "Wind Onshore [MWh] Berechnete Auflösungen": "Wind Onshore Production FC [MWh]",
                                                              "Photovoltaik [MWh] Berechnete Auflösungen": "Photovoltaik Production FC [MWh]",
                                                              "Sonstige [MWh] Berechnete Auflösungen": "Other Production FC [MWh]"})

load_forecast = load_forecast[[
    "Datum von",
    "Netzlast [MWh] Berechnete Auflösungen"]].rename(columns={"Datum von": "timestamp",
                                                              "Netzlast [MWh] Berechnete Auflösungen": "Total Load FC [MWh]",})

day_ahead_price = day_ahead_price[[
    "Datum von", "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen"]].rename(columns={
    "Datum von": "timestamp",
    "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen": "Day Ahead Price [EUR/MWh]"})

## Prepare data
# Datetime Objects
generation_forecast["timestamp"] = pd.to_datetime(generation_forecast["timestamp"], dayfirst=True)
load_forecast["timestamp"] = pd.to_datetime(load_forecast["timestamp"], dayfirst=True)
day_ahead_price["timestamp"] = pd.to_datetime(day_ahead_price["timestamp"], dayfirst=True)

## Merge Tables (each doubled hour is multiplied 4 times, but will be handled later on anyway with the same results)
df_join = pd.merge(pd.merge(load_forecast, generation_forecast, on="timestamp", how="inner"), day_ahead_price,on="timestamp",how="inner")

# Numeric columns are created
df_cleaned = df_join.replace("-", np.nan)
cols = df_cleaned.columns[1:]
for col in cols:
    s = (
        df_cleaned[col]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )

    # Convert literal "nan" or empty strings to real NaN
    s = s.replace(["nan", ""], pd.NA)

    # Now safely convert to numeric
    df_cleaned[col] = pd.to_numeric(s, errors="coerce")
df_cleaned

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
0,2018-10-01 02:00:00,42628.00,1750.50,4152.00,0.00,37311.50,51.41
1,2018-10-01 03:00:00,42986.75,1895.25,4436.25,0.00,36318.50,47.38
2,2018-10-01 04:00:00,44675.00,2138.25,4816.25,0.00,37481.50,47.59
3,2018-10-01 05:00:00,48813.25,2368.50,5276.00,0.00,41073.50,51.61
4,2018-10-01 06:00:00,57869.00,2649.25,5625.25,0.00,45581.50,69.13
...,...,...,...,...,...,...,...
67337,2026-06-04 19:00:00,52194.28,4978.72,22366.01,3950.46,25564.73,114.17
67338,2026-06-04 20:00:00,51930.65,5167.91,21943.25,1137.85,25791.74,117.96
67339,2026-06-04 21:00:00,50551.84,5451.33,22037.07,104.24,25573.48,115.37
67340,2026-06-04 22:00:00,48818.86,5682.32,22200.84,0.00,23723.35,111.69


### Data Quality Handling
1. Time shifts (missing and doubled hours)
2. Missing Values
3. Heteroscedastic Variance of Day Ahead Price over the time series.

These quality issues are handled in the following three subsections.

#### Time Shifts

For the time shifts of European time, two central problems appear in our data:
1. Doubled hours, when time is shifted backwards
    - The doubled hour is averaged out of the data (orientation for problem handling by Ziel & Weron, 2018)
2. Missing hours, when time is shifted forwards
    - The missing hour is forward filled (orientation for problem handling by Ziel & Weron, 2018)


Let's take a look at the doubled hours.

In [3]:
# look at duplicated timestamps
dups_ts = (
    df_cleaned["timestamp"]
    .value_counts()
    .loc[lambda x: x > 1]
    .index
)

df_cleaned.loc[df_cleaned["timestamp"].isin(dups_ts)]

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
648,2018-10-28 02:00:00,NaN,2621.00,10032.50,0.0,34983.50,41.62
649,2018-10-28 02:00:00,NaN,2621.00,10032.50,0.0,34983.50,41.59
650,2018-10-28 02:00:00,NaN,2524.00,10562.00,0.0,34744.00,41.62
651,2018-10-28 02:00:00,NaN,2524.00,10562.00,0.0,34744.00,41.59
652,2018-10-28 02:00:00,NaN,2621.00,10032.50,0.0,34983.50,41.62
...,...,...,...,...,...,...,...
62013,2025-10-26 02:00:00,40283.49,5662.18,33919.70,0.0,16569.90,2.02
62014,2025-10-26 02:00:00,39882.12,5739.19,33457.25,0.0,17222.03,3.19
62015,2025-10-26 02:00:00,39882.12,5739.19,33457.25,0.0,17222.03,2.02
62016,2025-10-26 02:00:00,39882.12,5662.18,33919.70,0.0,16569.90,3.19


Take-Away: each duplicated hour appears 8 times due to the merge process. The data handling of doubled hours will solve this problem.

In [4]:
## Handling of doubled hours via mean
df_time_cleaned = (
    df_cleaned
    .groupby("timestamp", as_index=False)
    .mean(numeric_only=True)
)
df_time_cleaned.loc[df_time_cleaned["timestamp"].isin(dups_ts)]

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
648,2018-10-28 02:00:00,NaN,2572.500,10297.250,0.0,34863.750,41.605
9383,2019-10-27 02:00:00,38813.750,5672.375,27504.125,0.0,19140.000,-19.970
18118,2020-10-25 02:00:00,38046.750,5299.000,24125.750,0.0,19260.250,0.120
27021,2021-10-31 02:00:00,42149.125,2995.625,16252.500,0.0,20583.375,66.760
35756,2022-10-30 02:00:00,42986.750,1517.750,7816.625,0.0,25391.625,100.060
44491,2023-10-29 02:00:00,37033.625,6074.750,23126.500,0.0,NaN,0.015
53226,2024-10-27 02:00:00,37638.875,2071.375,9575.750,0.0,NaN,81.330
61961,2025-10-26 02:00:00,40082.805,5700.685,33688.475,0.0,16895.965,2.605


Let's take a look at the missing hours.

In [5]:
# keep original timestamps to detect newly inserted rows later
original_timestamps = df_time_cleaned["timestamp"].copy()

# create a complete hourly time index
full_range = pd.date_range(
    start=df_time_cleaned["timestamp"].min(),
    end=df_time_cleaned["timestamp"].max(),
    freq="h"
)

# reindex to full hourly range (introduces NaNs for missing timestamps)
df_time_cleaned = (
    df_time_cleaned
    .set_index("timestamp")
    .reindex(full_range)
)
df_time_cleaned.index.name = "timestamp"

# identify rows that did not exist before (DST gaps)
new_rows = ~df_time_cleaned.index.isin(original_timestamps)

# look at missing timestamps
df_time_cleaned[~df_time_cleaned.index.isin(original_timestamps)]

,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
timestamp,,,,,,
2019-03-31 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2020-03-29 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2021-03-28 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-27 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2023-03-26 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-31 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2025-03-30 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-29 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# forward-fill all values
df_ffill = df_time_cleaned.ffill()

# fill only the newly inserted timestamps with forward-filled values
df_time_cleaned.loc[new_rows, :] = df_ffill.loc[new_rows, :]

# restore timestamp as a column
df_time_cleaned = df_time_cleaned.reset_index()

# look at missing timestamps
df_time_cleaned[~df_time_cleaned.index.isin(original_timestamps)]

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
0,2018-10-01 02:00:00,42628.00,1750.50,4152.00,0.00,37311.50,51.41
1,2018-10-01 03:00:00,42986.75,1895.25,4436.25,0.00,36318.50,47.38
2,2018-10-01 04:00:00,44675.00,2138.25,4816.25,0.00,37481.50,47.59
3,2018-10-01 05:00:00,48813.25,2368.50,5276.00,0.00,41073.50,51.61
4,2018-10-01 06:00:00,57869.00,2649.25,5625.25,0.00,45581.50,69.13
...,...,...,...,...,...,...,...
67289,2026-06-04 19:00:00,52194.28,4978.72,22366.01,3950.46,25564.73,114.17
67290,2026-06-04 20:00:00,51930.65,5167.91,21943.25,1137.85,25791.74,117.96
67291,2026-06-04 21:00:00,50551.84,5451.33,22037.07,104.24,25573.48,115.37
67292,2026-06-04 22:00:00,48818.86,5682.32,22200.84,0.00,23723.35,111.69


Take-Away: Missing and double hours due to time shifting are eliminated from the dataset in such a way that all timestamp are completely unique.

#### Missing Values

For the possible imputation of values, we introduce some seasonality features into the data set through feature engineering.

In [7]:
## first feature engineering
# Holidays
de = holidays.DE()
lu = holidays.LU()
df_time_cleaned["date only"] =  df_time_cleaned["timestamp"].dt.date
df_time_cleaned["German_holiday"] = df_time_cleaned["date only"].apply(lambda x: x in de)
df_time_cleaned["Luxembourg_holiday"] = df_time_cleaned["date only"].apply(lambda x: x in lu)
df_time_cleaned.drop(columns=["date only"], inplace=True)

# Timestamp patterns
df_time_cleaned["weekday"] = df_time_cleaned["timestamp"].dt.weekday.astype("category")
df_time_cleaned["hour"] = df_time_cleaned["timestamp"].dt.hour.astype("category")
df_time_cleaned["month"] = df_time_cleaned["timestamp"].dt.month.astype("category")
df_time_cleaned["year"] = df_time_cleaned["timestamp"].dt.year.astype("int")

In [131]:
# check for empty values
df_time_cleaned.isna().sum()

timestamp                               0
Total Load FC [MWh]                  1033
Wind Offshore Production FC [MWh]       0
Wind Onshore Production FC [MWh]        3
Photovoltaik Production FC [MWh]        3
Other Production FC [MWh]            1688
Day Ahead Price [EUR/MWh]               0
German_holiday                          0
Luxembourg_holiday                      0
weekday                                 0
hour                                    0
month                                   0
year                                    0
dtype: int64

To address the missing values in all 4 features, we perform data imputation using the kNN method. This way, the empty values are set to a more realistic value compared to applications of forward fill or other methods, especially when multiple values are missing continuously.

In [8]:
# define the columns which have empty values
impute_targets = df_time_cleaned.columns[df_time_cleaned.isna().any()]

# define the columns used for prediction
predictor_cols = df_time_cleaned.columns[
    ~df_time_cleaned.columns.isin(["timestamp","Day Ahead Price [EUR/MWh]"])
]

# working copy
df = df_time_cleaned.copy()
# cache for fitted models
model_cache = {}

# imputation by kNN
for target in impute_targets:

    # do not include target column in predictors
    candidate_predictors = [col for col in predictor_cols if col != target]

    # get all rows where the target is missing
    missing_indices = df_time_cleaned.index[df_time_cleaned[target].isna()]

    if len(missing_indices) == 0:
        print(f"{target}: no missing values")
        continue

    # iterate over empty rows
    for idx in missing_indices:

        # get all available predictors (non-NaN) for this row
        available_predictors = [
            col for col in candidate_predictors
            if pd.notna(df_time_cleaned.loc[idx, col])
        ]

        # at least one predictor must be available
        if len(available_predictors) == 0:
            continue

        predictors = tuple(available_predictors)

        # training rows: target present + predictors present
        train_mask = (
            df_time_cleaned[target].notna()
            & df_time_cleaned[list(predictors)].notna().all(axis=1)
        )

        X_train = df_time_cleaned.loc[train_mask, list(predictors)]
        y_train = df_time_cleaned.loc[train_mask, target]
        # skip when not enough train data is available
        if len(X_train) < 2:
            continue

        # build model if not cached
        cache_key = (target, predictors)

        if cache_key not in model_cache:
            k = min(5, len(X_train))
            knn = Pipeline([
                ("scaler", StandardScaler()),
                ("knn", KNeighborsRegressor(
                    n_neighbors=k,
                    weights="distance"
                ))
            ])
            knn.fit(X_train, y_train)
            model_cache[cache_key] = knn

        # predict with kNN
        X_pred = df_time_cleaned.loc[[idx], list(predictors)]
        prediction = model_cache[cache_key].predict(X_pred)[0]


        df.loc[idx, target] = prediction

print(df[impute_targets].isna().sum())

Total Load FC [MWh]                 0
Wind Onshore Production FC [MWh]    0
Photovoltaik Production FC [MWh]    0
Other Production FC [MWh]           0
dtype: int64


Take Away: All missing values are eliminated.

#### Variance Stabilization

As energy time-series have the potential to have heteroscedastic variances over time (Ziel and Weron, 2018), we take a look at the time-series to analyze the need for stabilization.

In [9]:
# select price column
col = "Day Ahead Price [EUR/MWh]"

# build line plot before price stabilization
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(df["timestamp"], df[col], color="royalblue", linewidth=2)

ax.set_title(f"{col} prior variance stabilization", fontsize=16, pad=12)
ax.set_xlabel("time")
ax.set_ylabel(col)

plt.tight_layout()
fname = f"plots/Day_Ahead_Price_EUR_MWh_prior_stabilization.png"

plt.savefig(fname, dpi=100, bbox_inches="tight")
plt.close()
print(f'{col} saved \u2192 {fname}')

Day Ahead Price [EUR/MWh] saved → plots/Day_Ahead_Price_EUR_MWh_prior_stabilization.png


Take-Away:
- Heteroscedastic variance over time series are apparent
- This needs to be handled via variance stabilization

Let's stabilize the prices based on the method given by Ziel and Weron (2018).

In [10]:
# select the price column
col = "Day Ahead Price [EUR/MWh]"

# sort data chronologically
df_sorted = df.sort_values("timestamp").reset_index(drop=True)

# set parameters for the first 730-day window
start_ts = df_sorted["timestamp"].min()
end_ts = start_ts + pd.Timedelta(days=730)

first_window = df_sorted[df_sorted["timestamp"] < end_ts]

# calculate median and MAD for the first window
a = first_window[col].median()
b = 1.4826 * np.median(np.abs(first_window[col] - a))

# create new column for stabilized price
df_sorted["Stabilized Day Ahead Price [EUR/MWh]"] = np.nan

# dictionary for later inverse transformation
transform_params = {}

# transform the first window
mask_first = df_sorted["timestamp"] < end_ts
df_sorted.loc[mask_first, "Stabilized Day Ahead Price [EUR/MWh]"] = np.arcsinh(
    (df_sorted.loc[mask_first, col] - a) / b
)

# save parameters for all timestamps of the first window (for later inverse transformation)
first_days = df_sorted.loc[mask_first, "timestamp"].dt.floor("D").unique()
for d in first_days:
    transform_params[pd.Timestamp(d)] = (a, b)

# iterate over all subsequent days and apply rolling window transformation
unique_days = df_sorted["timestamp"].dt.floor("D").unique()
window_days = 730

for i in range(window_days, len(unique_days)):

    # train window
    train_start = unique_days[i - window_days]
    train_end   = unique_days[i]

    train = df_sorted[(df_sorted["timestamp"] >= train_start) &
                      (df_sorted["timestamp"] <  train_end)]

    # parameters
    a_i = train[col].median()
    b_i = 1.4826 * np.median(np.abs(train[col] - a_i))
    if b_i == 0:
        b_i = 1e-6

    # test day
    test_day = unique_days[i]
    mask_test = df_sorted["timestamp"].dt.floor("D") == test_day

    # transform test day
    df_sorted.loc[mask_test, "Stabilized Day Ahead Price [EUR/MWh]"] = np.arcsinh(
        (df_sorted.loc[mask_test, col] - a_i) / b_i
    )

    # save parameters for the test day (for later inverse transformation)
    transform_params[pd.Timestamp(test_day)] = (a_i, b_i)

df = df_sorted.copy()

In [11]:
# select stabilized price column
col = df.columns[-1]

# build line plot after price stabilization
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(df["timestamp"], df[col], color="royalblue", linewidth=2)

ax.set_title(f"Day Ahead Price [EUR/MWh] after variance stabilization", fontsize=16, pad=12)
ax.set_xlabel("time")
ax.set_ylabel(col)

plt.tight_layout()
fname = f"plots/Day_Ahead_Price_EUR_MWh_after_stabilization.png"

plt.savefig(fname, dpi=100, bbox_inches="tight")
plt.close()
print(f'Day Ahead Price [EUR/MWh] after variance stabilization saved \u2192 {fname}')

Day Ahead Price [EUR/MWh] after variance stabilization saved → plots/Day_Ahead_Price_EUR_MWh_after_stabilization.png


Take-Away:
After the iterative application of the variance stabilization method based on MAD and median of the 730 day training phase as well as the asinh function, the time series chart of the DE-LU Day-Ahead Energy price looks way less spiky. Based on this, we will look at the distributions prior and after the variance stabilization.

In [12]:
# create histogram of day ahead price
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(
    df["Day Ahead Price [EUR/MWh]"],
    bins=100,
    color="steelblue",
    edgecolor="white",
    alpha=0.9
)

ax.set_title("Distribution of Day-Ahead Price [€/MWh]", fontsize=16, pad=12)
ax.set_xlabel("Day-Ahead Price [€/MWh]")
ax.set_ylabel("Count")

ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("plots/day_ahead_price_distribution_prior_stabilization.png", dpi=100, bbox_inches="tight")
plt.close()
print('Day-Ahead Price distribution prior stabilization saved \u2192 plots/day_ahead_price_distribution_prior_stabilization.png')

Day-Ahead Price distribution prior stabilization saved → plots/day_ahead_price_distribution_prior_stabilization.png


In [13]:
# create histogram of stabilized day ahead price
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(
    df["Stabilized Day Ahead Price [EUR/MWh]"],
    bins=100,
    color="steelblue",
    edgecolor="white",
    alpha=0.9
)

ax.set_title("Distribution of Stabilized Day-Ahead Price [€/MWh]", fontsize=16, pad=12)
ax.set_xlabel("Stabilized Day-Ahead Price [€/MWh]")
ax.set_ylabel("Count")

ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("plots/stabilized_day_ahead_price_distribution.png", dpi=100, bbox_inches="tight")
plt.close()
print('Stabilized Day-Ahead Price distribution saved \u2192 plots/stabilized_day_ahead_price_distribution.png')

Stabilized Day-Ahead Price distribution saved → plots/stabilized_day_ahead_price_distribution.png


Take-Away:
- Distribution of prices is rather left skewed on the positive side of 0.
- positive and negative price outliers exist with positive outliers being more frequent than negative outliers.

### Additional Feature Engineering

For the supervised learning part, some lag features are introduced which can be used in all different models, but especially in the Autoregressive with Exogenous Input Model.

In [16]:
# set column names
timestamp_col = "timestamp"
target_col = "Stabilized Day Ahead Price [EUR/MWh]"

# prepare timestamp and sort
df[timestamp_col] = pd.to_datetime(df[timestamp_col])
df = df.sort_values(timestamp_col).reset_index(drop=True)
df["date"] = df[timestamp_col].dt.floor("D")

# pivot prices by date and hour
price_by_day_hour = (
    df.pivot_table(
        index="date",
        columns="hour",
        values=target_col,
        aggfunc="first"
    )
    .sort_index()
)

# create lagged day-hour price features for last 7 days
lag_blocks = []

for d in range(1, 8):
    lag_block = price_by_day_hour.shift(d)
    lag_block.columns = [f"price_d-{d}_h{int(col)}" for col in lag_block.columns]
    lag_blocks.append(lag_block)

day_hour_lags = pd.concat(lag_blocks, axis=1)

# daily min/max from previous days
daily_stats = price_by_day_hour.agg(["min", "max"], axis=1)
daily_stats.columns = ["daily_min", "daily_max"]

daily_lag_blocks = []

for d in range(1, 8):
    daily_lag = daily_stats.shift(d).copy()
    daily_lag.columns = [f"{col}_d-{d}" for col in daily_lag.columns]
    daily_lag_blocks.append(daily_lag)

daily_lags = pd.concat(daily_lag_blocks, axis=1)

# combine all lag features
lag_features = pd.concat([day_hour_lags, daily_lags], axis=1)

# merge back to original df
df_final = df.merge(lag_features, left_on="date", right_index=True, how="left")

# cleanup
df_final = df_final.drop(columns=["date"])
df_final = df_final.dropna().reset_index(drop=True)

df_final

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh],German_holiday,Luxembourg_holiday,weekday,...,daily_min_d-3,daily_max_d-3,daily_min_d-4,daily_max_d-4,daily_min_d-5,daily_max_d-5,daily_min_d-6,daily_max_d-6,daily_min_d-7,daily_max_d-7
0,2018-10-09 00:00:00,47590.250000,3053.00,4436.75,0.00,44412.25,56.77,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
1,2018-10-09 01:00:00,46000.750000,3110.00,4333.50,0.00,43724.50,57.43,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
2,2018-10-09 02:00:00,45662.512065,3180.00,4231.00,0.00,43023.00,55.11,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
3,2018-10-09 03:00:00,45362.156474,3249.25,4217.00,0.00,42405.75,52.35,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
4,2018-10-09 04:00:00,47994.961623,3308.25,4233.75,0.00,43680.00,54.86,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67099,2026-06-04 19:00:00,52194.280000,4978.72,22366.01,3950.46,25564.73,114.17,False,False,3,...,-0.187284,2.713492,-1.724049,1.346221,-1.740193,1.55129,-1.727570,2.153582,-1.728640,2.861263
67100,2026-06-04 20:00:00,51930.650000,5167.91,21943.25,1137.85,25791.74,117.96,False,False,3,...,-0.187284,2.713492,-1.724049,1.346221,-1.740193,1.55129,-1.727570,2.153582,-1.728640,2.861263
67101,2026-06-04 21:00:00,50551.840000,5451.33,22037.07,104.24,25573.48,115.37,False,False,3,...,-0.187284,2.713492,-1.724049,1.346221,-1.740193,1.55129,-1.727570,2.153582,-1.728640,2.861263
67102,2026-06-04 22:00:00,48818.860000,5682.32,22200.84,0.00,23723.35,111.69,False,False,3,...,-0.187284,2.713492,-1.724049,1.346221,-1.740193,1.55129,-1.727570,2.153582,-1.728640,2.861263


### Data Investigation

Prior to the data modeling part, the data is investigated using correlation matrix and time-series plots.

#### Correlation Matrix

In [77]:
# build correlation matrix
cols = df.columns.drop("timestamp")[:13]

corr_matrix = df[cols].corr()

# build the plot
plt.figure(figsize=(16, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"}
)
plt.title("Correlation Matrix", fontsize=18, pad=20)
plt.tight_layout()
plt.savefig('plots/corr_matrix.png', dpi=100, bbox_inches='tight')
plt.close()
print('Correlation matrix saved \u2192 plots/corr_matrix.png')

Correlation matrix saved → plots/corr_matrix.png


Take-Away:
- The two wind production forecasts have a strong positive correlation, which is expected given that both are driven by similar meteorological conditions.
- Load and total production are positively correlated, reflecting typical market dynamics: higher demand requires higher generation levels.
- The positive correlation between day‑ahead prices and conventional (non‑renewable) production suggests that prices tend to rise when more traditional generation is required.
- The slightly positive correlation between the year variable and renewable production indicates a structural increase in renewable generation capacity over time.
- This trend is underlined by the negative correlation between the year variable and conventional production.
- The positive correlation between day‑ahead prices and the year variable points to a long‑term upward trend in electricity prices, potentially driven by inflation, geopolitical events, and broader market disruptions such as wars or pandemics.


#### Time-Series Plots

In [78]:
plt.style.use("seaborn-v0_8-whitegrid")

# restrict to relevant columns
cols = df.columns.drop("timestamp")[:4]

# build line plots iteratively
for col in cols:
    fig, ax = plt.subplots(figsize=(16, 4))

    ax.plot(df["timestamp"], df[col], color="royalblue", linewidth=2)

    ax.set_title(f"{col} over time", fontsize=16, pad=12)
    ax.set_xlabel("time")
    ax.set_ylabel(col)

    plt.tight_layout()
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", col).strip("_")
    fname = f"plots/{safe_name}_over_time.png"

    plt.savefig(fname, dpi=100, bbox_inches="tight")
    plt.close()
    print(f'{col} saved \u2192 {fname}')

Total Load FC [MWh] saved → plots/Total_Load_FC_MWh_over_time.png
Wind Offshore Production FC [MWh] saved → plots/Wind_Offshore_Production_FC_MWh_over_time.png
Wind Onshore Production FC [MWh] saved → plots/Wind_Onshore_Production_FC_MWh_over_time.png
Photovoltaik Production FC [MWh] saved → plots/Photovoltaik_Production_FC_MWh_over_time.png


Take-Away:
- The time-series plots indicate expected seasonality patterns for load and production figures.
- The Price time series as underlines the slightly increasing price mean over time.
- Also, the price variance gets larger as the years go on, this is why the the variance stabilization is applied.
- The variance stabilization is shown in the second price time series.

### Negative Price Distribution

In [79]:
# marker if price is negative or positive
df["negative_price_flag"] = (df["Day Ahead Price [EUR/MWh]"] < 0).astype(int)

# build distribution by hour of day
neg_dist = df.groupby(["hour", "negative_price_flag"], observed=False).size().unstack()
neg_dist = neg_dist.rename(columns={
    0: "Positive prices",
    1: "Negative prices"
})


# convert to long format for plotly
neg_dist_plot = neg_dist.reset_index().melt(
    id_vars="hour",
    value_vars=["Positive prices", "Negative prices"],
    var_name="type",
    value_name="count"
)

hours = neg_dist.index
pos = neg_dist["Positive prices"]
neg = neg_dist["Negative prices"]

# stacked bar plot for positive and negative price distribution by hour
plt.figure(figsize=(14, 5))
plt.bar(hours, pos, label="Positive prices", color="steelblue")
plt.bar(hours, neg, bottom=pos, label="Negative prices", color="firebrick")

plt.title("Positive and Negative Prices by Hour", fontsize=16, pad=12)
plt.xlabel("Hour of Day")
plt.ylabel("Number of Observations")
plt.xticks(range(24))
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend()

plt.tight_layout()
plt.savefig("plots/negative_price_distribution.png", dpi=100, bbox_inches="tight")
plt.close()

print('Negative Price distribution saved \u2192 plots/negative_price_distribution.png')

# drop column again
df.drop(columns=["negative_price_flag"], inplace=True)

Negative Price distribution saved → plots/negative_price_distribution.png


Take-Away:
- Negative prices seem to occur mostly around midday which suits the thesis that renewable energies, especially Photovoltaik are causing this phenomenon

---
## Section 2 — Causal Inference Block - TODO

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] A causal graph (DAG) is constructed and the assumed relationships are justified
- [ ] Identification strategy is appropriate (backdoor / IV / propensity score)
- [ ] Estimation is implemented correctly using DoWhy (or equivalent)
- [ ] At least one refutation test is run and its result is interpreted

---
*This example uses backdoor adjustment (linear regression). Replace with the strategy appropriate for your research question.*

In [142]:
G = nx.DiGraph()

solid_edges = [
    # Demand side / Load forecast
    ('seasonality', 'load_forecast'),
    ('holidays', 'load_forecast'),

    # Supply side
    ('forecasted_supply_renewables', 'forecasted_supply_traditional_powerplants'),
    ('forecasted_supply_renewables', 'supply_forecast'),
    ('forecasted_supply_traditional_powerplants', 'supply_forecast'),

    # Price formation
    ('load_forecast', 'day_ahead_price'),
    ('supply_forecast', 'day_ahead_price'),
]

dashed_edges = [
    ('seasonality', 'weather'),
    ('weather', 'load_forecast'),
    ('weather', 'forecasted_supply_renewables'),
    ('holidays', 'industrial_activities'),
    ('industrial_activities', 'load_forecast'),
    ('downtimes_powerplants', 'forecasted_supply_traditional_powerplants'),
    ('prices_alternative_energy_sources', 'forecasted_supply_traditional_powerplants'),
]

G.add_edges_from(solid_edges + dashed_edges)

pos = {
    'holidays': (0.0, -1.0),
    'industrial_activities': (0.0, 1.0),

    'seasonality': (1.8, 1.5),
    'weather': (3.4, 1.5),

    'load_forecast': (2.4, 0.0),
    'day_ahead_price': (4.4, 0.0),

    'prices_alternative_energy_sources': (6.4, -1.0),

    'supply_forecast': (6.4, 0.0),
    'forecasted_supply_renewables': (7.8, 1.5),
    'forecasted_supply_traditional_powerplants': (7.8, 0.0),
    'downtimes_powerplants': (7.8, -1.0),
}

labels = {
    'seasonality': 'Seasonality',
    'weather': 'Weather',
    'holidays': 'Holidays',
    'industrial_activities': 'Industrial\nActivities',
    'load_forecast': 'Load\nForecast',
    'supply_forecast': 'Supply\nForecast',
    'forecasted_supply_renewables': 'Forecasted Supply\nrRenewables',
    'forecasted_supply_traditional_powerplants': 'Forecasted Supply\nTraditional\nPowerplants',
    'downtimes_powerplants': 'Downtimes of\nPowerplants',
    'prices_alternative_energy_sources': 'Prices of alternative\nEnergy Sources',
    'day_ahead_price': 'Day-Ahead\nPrice'
}



node_colors = [
    '#FDEBD0' if n in (
        'seasonality',
        'weather',
        'holidays',
        'industrial_activities',
        'prices_alternative_energy_sources',
        'downtimes_powerplants'
    ) else
    '#F9E79F' if n in (
        'load_forecast',
        'supply_forecast',
        'forecasted_supply_renewables',
        'forecasted_supply_traditional_powerplants'
    ) else
    '#A9DFBF'
    for n in G.nodes()
]

 
fig, ax = plt.subplots(figsize=(14, 6))

# Nodes zeichnen
nx.draw_networkx_nodes(
    G,
    pos=pos,
    ax=ax,
    node_color=node_colors,
    node_size=5000,
    edgecolors='dimgray',
    linewidths=1,
    margins=0.1
)

# Labels zeichnen
nx.draw_networkx_labels(
    G,
    pos=pos,
    labels=labels,
    ax=ax,
    font_size=7.5
)

# Solide gerichtete Kanten mit Pfeilen
nx.draw_networkx_edges(
    G,
    pos=pos,
    edgelist=solid_edges,
    ax=ax,
    arrows=True,
    arrowstyle='-|>',
    arrowsize=22,
    edge_color='dimgray',
    width=1.6,
    style='solid',
    min_source_margin=25,
    min_target_margin=30
)

# Gestrichelte gerichtete Kanten mit Pfeilen
nx.draw_networkx_edges(
    G,
    pos=pos,
    edgelist=dashed_edges,
    ax=ax,
    arrows=True,
    arrowstyle='-|>',
    arrowsize=22,
    edge_color='dimgray',
    width=1.6,
    style='dashed',
    min_source_margin=25,
    min_target_margin=30
)


ax.set_title('Causal DAG: Drivers of Day-Ahead Electricity Prices', fontsize=12)
ax.axis('off')

plt.tight_layout()
plt.savefig("plots/dag.png", dpi=150, bbox_inches='tight')
plt.close()


TODO: Interpretation

---
## Section 3 — Supervised Learning Block

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] Model choice is justified relative to the prediction task
- [ ] Train/test split and cross-validation are used correctly
- [ ] An appropriate metric is reported and interpreted (RMSE, AUC, accuracy, …)
- [ ] Results are compared to a baseline or alternative model

---
*This example predicts training participation (binary classification) from worker characteristics.
Replace with your own prediction task — classification or regression.*

### Modeling

In [19]:
# create one df per hour of day
hourly_dfs = {}

for hour in range(24):
    hourly_dfs[hour] = df_final[df_final["hour"] == hour].copy()

# look at one example
hourly_dfs[0].head()

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh],German_holiday,Luxembourg_holiday,weekday,...,daily_min_d-3,daily_max_d-3,daily_min_d-4,daily_max_d-4,daily_min_d-5,daily_max_d-5,daily_min_d-6,daily_max_d-6,daily_min_d-7,daily_max_d-7
0,2018-10-09,47590.250000,3053.00,4436.75,0.0,44412.25,56.77,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.997130,-1.773498,1.631451,-1.624609,1.602742
24,2018-10-10,47056.258035,1653.50,3492.50,0.0,45229.00,53.70,False,False,2,...,0.430020,1.855446,0.668194,1.883336,0.678337,1.897518,0.618903,1.997130,-1.773498,1.631451
48,2018-10-11,49332.677681,4619.25,18584.50,0.0,32970.25,35.01,False,False,3,...,0.716108,2.227772,0.430020,1.855446,0.668194,1.883336,0.678337,1.897518,0.618903,1.997130
72,2018-10-12,50265.750000,2824.25,12392.75,0.0,37070.00,45.85,False,False,4,...,0.969730,2.343601,0.716108,2.227772,0.430020,1.855446,0.668194,1.883336,0.678337,1.897518
96,2018-10-13,47562.500000,4277.25,13403.75,0.0,33339.00,47.80,False,False,5,...,-0.213461,2.088352,0.969730,2.343601,0.716108,2.227772,0.430020,1.855446,0.668194,1.883336


In [ ]:
# settings
SCORING = "neg_mean_absolute_error"
HOUR_N_JOBS = 12
GRID_N_JOBS = 2
RF_N_JOBS = 4
PERM_N_JOBS = 4
N_SPLITS = 3

GRID_SEARCH_SPACES = {
    
    "Decision Tree": {
        "estimator": DecisionTreeRegressor(
            random_state=42
        ),
        "param_grid": {
            "max_depth": [3, 5, 10, None],
            "min_samples_leaf": [1, 5, 10, 20],
            "min_samples_split": [2, 10, 20]
        }
    },

    "ARX": {
        "estimator": Pipeline(steps=[
            ("scaler", StandardScaler()),
            ("model", Lasso(
                random_state=42,
                max_iter=10000
            ))
        ]),
        "param_grid": {
            "model__alpha": [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
        }
    },

    "Random Forest": {
        "estimator": RandomForestRegressor(
            random_state=42,
            n_jobs=RF_N_JOBS
        ),
        "param_grid": {
            "n_estimators": [100, 300, 500],
            "max_depth": [5, 10, None],
            "min_samples_leaf": [1, 2, 5],
            "max_features": ["sqrt", 0.5, 1.0]
        }
    },

    "Neural Network": {
        "estimator": TransformedTargetRegressor(
            regressor=Pipeline(steps=[
                ("scaler", StandardScaler()),
                ("model", MLPRegressor(
                    activation="relu",
                    solver="adam",
                    max_iter=3000,
                    early_stopping=True,
                    n_iter_no_change=20,
                    random_state=42,
                    learning_rate="adaptive"
                ))
            ]),
            transformer=StandardScaler()
        ),
        "param_grid": {
            "regressor__model__hidden_layer_sizes": [
                (16,),
                (32,),
                (32, 16),
                (64, 32),
                (64, 32, 16),
                (128, 64, 32, 16, 8)
            ],
            "regressor__model__learning_rate_init": [
                0.001,
                0.0005,
                0.0001
            ],
            "regressor__model__alpha": [
                0.0001,
                0.001,
                0.01
            ]
        }
    }
}


# path for results
RESULTS = BASE / "results"

# initialize containers for all results
full_predictions = []
full_importances = []

target_col = "Stabilized Day Ahead Price [EUR/MWh]"


def run_hour(i):

    print(f"Start hour: {i}")

    # get current hour data frame
    df_model = hourly_dfs[i].copy()

    # sort by timestamp
    df_model = df_model.sort_values("timestamp").reset_index(drop=True)

    # set features
    feature_cols = df_model.columns[
        ~df_model.columns.isin([target_col, "timestamp", "Day Ahead Price [EUR/MWh]"])
    ].tolist()

    # set collector lists
    all_predictions = []
    all_feature_importances = []

    # determine first test day
    first_test_day = df_model["timestamp"].min() + pd.DateOffset(years=2)

    # set last possible test day
    last_test_day = pd.to_datetime("04.06.2026  00:00:00")
    
    # initialize current test day
    current_test_day = first_test_day

    # set hpo dicts
    hpo_done = False
    best_params_by_model = {}
    best_cv_score_by_model = {}
    best_cv_mae_by_model = {}

    # perform rolling forecast
    while current_test_day <= last_test_day:

        print(f"Hour {i} | Test day: {current_test_day.date()}")

        # train set: prior year
        train_start = current_test_day - pd.DateOffset(years=2)
        train_end = current_test_day

        train = df_model[
            (df_model["timestamp"] >= train_start) &
            (df_model["timestamp"] < train_end)
        ].copy()

        # test set: next day
        test_start = current_test_day
        test_end = current_test_day + pd.DateOffset(days=1)

        test = df_model[
            (df_model["timestamp"] >= test_start) &
            (df_model["timestamp"] < test_end)
        ].copy()

        # skip if no data
        if len(train) == 0 or len(test) == 0:
            current_test_day += pd.DateOffset(days=1)
            continue

        if len(train) == 0 or len(test) == 0:
            current_test_day += pd.DateOffset(days=1)
            continue

        X_train = train[feature_cols]
        y_train = train[target_col]

        X_test = test[feature_cols]
        y_test = test[target_col]

        # hpo on first training intervall
        if not hpo_done:

            print(
                f"Hour {i} | Running HPO only once on first valid interval: "
                f"{current_test_day.date()}"
            )

            # time series cross-validation only on first training set
            tscv = TimeSeriesSplit(n_splits=N_SPLITS)

            for model_name, setup in GRID_SEARCH_SPACES.items():

                estimator = setup["estimator"]
                param_grid = setup["param_grid"]

                grid_search = GridSearchCV(
                    estimator=estimator,
                    param_grid=param_grid,
                    scoring=SCORING,
                    cv=tscv,
                    n_jobs=GRID_N_JOBS,
                    refit=True
                )

                grid_search.fit(X_train, y_train)

                best_params_by_model[model_name] = grid_search.best_params_
                best_cv_score_by_model[model_name] = grid_search.best_score_
                best_cv_mae_by_model[model_name] = -grid_search.best_score_

                print(
                    f"Hour {i} | {model_name} | "
                    f"Best CV MAE: {-grid_search.best_score_:.4f} | "
                    f"Best params: {grid_search.best_params_}"
                )

            hpo_done = True

        # train models with optimized hpos
        for model_name, setup in GRID_SEARCH_SPACES.items():

            estimator = setup["estimator"]

            # fresh estimator copy for each rolling window
            model = clone(estimator)

            # use fixed parameters from first HPO interval
            model.set_params(**best_params_by_model[model_name])

            # fit model on current rolling training window
            model.fit(X_train, y_train)

            # predict
            y_pred = model.predict(X_test)

            # save predictions
            pred_df = pd.DataFrame({
                "timestamp": test["timestamp"].values,
                "model": model_name,
                "hour": i,
                "y_true": y_test.values,
                "y_pred": y_pred,
                "train_start": train_start,
                "train_end": train_end,
                "test_day": test_start,
                "train_rows": len(train),
                "test_rows": len(test),
                "best_cv_score_neg_mae": best_cv_score_by_model[model_name],
                "best_cv_mae": best_cv_mae_by_model[model_name],
                "best_params": str(best_params_by_model[model_name])
            })

            all_predictions.append(pred_df)

            # Feature Importance
            if model_name in ["Random Forest", "Decision Tree"]:

                importance_values = model.feature_importances_

                fi_df = pd.DataFrame({
                    "test_day": test_start,
                    "model": model_name,
                    "hour": i,
                    "feature": feature_cols,
                    "importance": importance_values,
                    "importance_type": "tree_feature_importance",
                    "best_params": str(best_params_by_model[model_name]),
                    "best_cv_mae": best_cv_mae_by_model[model_name]
                })

                all_feature_importances.append(fi_df)

            elif model_name == "Neural Network":

                # permutation importance for neural network
                perm = permutation_importance(
                    model,
                    X_train,
                    y_train,
                    random_state=42,
                    scoring="neg_mean_absolute_error",
                    n_jobs=PERM_N_JOBS
                )

                fi_df = pd.DataFrame({
                    "test_day": test_start,
                    "model": model_name,
                    "hour": i,
                    "feature": feature_cols,
                    "importance": perm.importances_mean,
                    "importance_std": perm.importances_std,
                    "importance_type": "permutation_importance_train",
                    "best_params": str(best_params_by_model[model_name]),
                    "best_cv_mae": best_cv_mae_by_model[model_name]
                })

                all_feature_importances.append(fi_df)

        # Naive Baseline
        naive_prediction = y_train.iloc[-1]
        y_pred_naive = np.repeat(naive_prediction, len(y_test))

        pred_df_naive = pd.DataFrame({
            "timestamp": test["timestamp"].values,
            "model": "Naive",
            "hour": i,
            "y_true": y_test.values,
            "y_pred": y_pred_naive,
            "train_start": train_start,
            "train_end": train_end,
            "test_day": test_start,
            "train_rows": len(train),
            "test_rows": len(test),
            "best_cv_score_neg_mae": np.nan,
            "best_cv_mae": np.nan,
            "best_params": "not_applicable"
        })

        all_predictions.append(pred_df_naive)

        fi_naive = pd.DataFrame({
            "test_day": test_start,
            "model": "Naive",
            "hour": i,
            "feature": feature_cols,
            "importance": np.nan,
            "importance_type": "not_applicable",
            "best_params": "not_applicable",
            "best_cv_mae": np.nan
        })

        all_feature_importances.append(fi_naive)

        # go to next day
        current_test_day += pd.DateOffset(days=1)

    # get all predictions together for current hour
    rolling_predictions = pd.concat(all_predictions, ignore_index=True)

    # get all importances together for current hour
    feature_importances = pd.concat(all_feature_importances, ignore_index=True)

    # save results as csv
    feature_importances.to_csv(
        RESULTS / f"importances_{i}.csv",
        index=False
    )

    rolling_predictions.to_csv(
        RESULTS / f"predictions_{i}.csv",
        index=False
    )

    print(f"Finished hour: {i}")

    return {
        "hour": i,
        "predictions_path": str(RESULTS / f"predictions_{i}.csv"),
        "importances_path": str(RESULTS / f"importances_{i}.csv")
    }


# run all 24 hours in parallel
with threadpool_limits(limits=1):
    hour_results = Parallel(
        n_jobs=HOUR_N_JOBS,
        backend="loky",
        verbose=10
    )(
        delayed(run_hour)(i) for i in range(0, 24)
    )

hour_results

Start hour: 0


### Load the results from the .csv files to save time.

In [25]:
BASE = Path(".")
RESULTS = BASE / "results"

# load predictions from csv results
pred_list = []
for i in range(24):
    df = pd.read_csv(
        RESULTS / f"predictions_{i}.csv",
        parse_dates=["timestamp", "train_start", "train_end", "test_day"]
    )
    pred_list.append(df)

full_predictions = pd.concat(pred_list, ignore_index=True)

# load importance from csv results
imp_list = []
for i in range(24):
    df = pd.read_csv(
        RESULTS / f"importances_{i}.csv",
        parse_dates=["test_day"]
    )
    imp_list.append(df)

full_importances = pd.concat(imp_list, ignore_index=True) 


,timestamp,model,hour,y_true,y_pred,train_start,train_end,test_day,train_rows,test_rows,best_cv_score_neg_mae,best_cv_mae,best_params
0,2020-10-09,Decision Tree,0,-1.068175,-1.089962,2018-10-09,2020-10-09,2020-10-09,731,1,-0.021261,0.021261,"{'max_depth': 10, 'min_samples_leaf': 5, 'min_..."
1,2020-10-09,ARX,0,-1.068175,-1.129451,2018-10-09,2020-10-09,2020-10-09,731,1,-0.042106,0.042106,{'model__alpha': 0.001}
2,2020-10-09,Random Forest,0,-1.068175,-1.077467,2018-10-09,2020-10-09,2020-10-09,731,1,-0.014521,0.014521,"{'max_depth': 10, 'max_features': 1.0, 'min_sa..."
3,2020-10-09,Neural Network,0,-1.068175,-1.158359,2018-10-09,2020-10-09,2020-10-09,731,1,-0.138473,0.138473,"{'regressor__model__alpha': 0.0001, 'regressor..."
4,2020-10-09,Naive,0,-1.068175,-1.039719,2018-10-09,2020-10-09,2020-10-09,731,1,NaN,NaN,not_applicable


In [26]:
def inverse_stabilization(y_pred_transformed, test_day, transform_params):
    """
    Inverse transformation of variance stabilization to compare tim-series in original data format:
        P = b * sinh(Y_hat) + a
    """
    a_i, b_i = transform_params[pd.Timestamp(test_day)]
    return b_i * np.sinh(y_pred_transformed) + a_i


In [46]:
## Application of inverse transformation

preds_real = full_predictions.copy()

preds_real["y_pred_real"] = np.nan
preds_real["y_true_real"] = np.nan

for idx, row in preds_real.iterrows():

    # get test day
    test_day = pd.Timestamp(row["test_day"]).floor("D")

    # get transformed values
    y_pred_t = row["y_pred"]
    y_true_t = row["y_true"]

    # get parameters of transformation
    if test_day not in transform_params:
        # if a day is missing -> skip
        continue

    a_i, b_i = transform_params[test_day]

    # inverse transformation
    preds_real.at[idx, "y_pred_real"] = b_i * np.sinh(y_pred_t) + a_i
    preds_real.at[idx, "y_true_real"] = b_i * np.sinh(y_true_t) + a_i


### Performance Evaluation

In [28]:
# generate model performance on original scale
model_performance_real = (
    preds_real
    .groupby(["model", "hour"])
    .apply(lambda x: pd.Series({
        "MAE": mean_absolute_error(x["y_true_real"], x["y_pred_real"]),
        "RMSE": np.sqrt(mean_squared_error(x["y_true_real"], x["y_pred_real"])),
        "R2": r2_score(x["y_true_real"], x["y_pred_real"]),
        "Observations": len(x)
    }))
    .reset_index()
    .sort_values("MAE")
)
model_performance_real

,model,hour,MAE,RMSE,R2,Observations
3,ARX,3,2.215808,3.408892,0.992346,2006.0
99,Random Forest,3,2.308732,3.616395,0.991386,2006.0
2,ARX,2,2.407793,3.718610,0.990858,2006.0
98,Random Forest,2,2.467229,3.824166,0.990332,2006.0
4,ARX,4,2.501997,3.826647,0.990859,2006.0
...,...,...,...,...,...,...
54,Naive,6,26.233986,37.981477,0.412819,2006.0
58,Naive,10,26.437589,39.021441,0.529538,2006.0
57,Naive,9,28.729867,42.384042,0.493056,2006.0
55,Naive,7,29.860863,44.148956,0.416324,2006.0


In [29]:
# combine all predictions into one DataFrame
full_predictions_df = preds_real.copy()
full_predictions_df = full_predictions_df.sort_values("timestamp").reset_index(drop=True)

model_name = "Random Forest"

# filter for the selected model
temp = full_predictions_df[full_predictions_df["model"] == model_name]
temp = temp.sort_values("timestamp")

plt.figure(figsize=(18,6))

# ACT prices
plt.plot(
    temp["timestamp"],
    temp["y_true_real"],     
    label="Actual (real scale)",
    color="black",
    alpha=0.7
)

# FC prices
plt.plot(
    temp["timestamp"],
    temp["y_pred_real"],     
    label="Predicted (real scale)",
    color="royalblue",
    alpha=0.7
)

# time-series plot
plt.title(f"Full Time Series – {model_name} (Real Scale)")
plt.xlabel("Timestamp")
plt.ylabel("Pice Day Ahead [€/MWh]")  
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig('plots/time_series_comparison.png', dpi=100, bbox_inches='tight')
plt.close()

print('Comparison saved → plots/time_series_comparison.png')


Comparison saved → plots/time_series_comparison.png


In [30]:
# create a plot per model
for model_name in model_performance_real["model"].unique():

    temp = model_performance_real[
        model_performance_real["model"] == model_name
    ].copy()

    # sort by hour
    temp = temp.sort_values("hour")

    # get mean value
    mae_mean = temp["MAE"].mean()

    # plot
    plt.figure(figsize=(12, 5))

    bars = plt.bar(
        temp["hour"],
        temp["MAE"]
    )

    # data lables
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    # mean line
    plt.plot(
        temp["hour"],
        [mae_mean] * len(temp),
        linestyle="--",
        linewidth=2.5,
        label=f"Mean MAE: {mae_mean:.2f}"
    )

    # settings
    plt.title(f"MAE per Hour – {model_name}")
    plt.xlabel("Hour of Day")
    plt.ylabel("MAE [€/MWh]")

    plt.xticks(temp["hour"])
    plt.grid(axis="y", alpha=0.3)
    plt.legend()

    plt.savefig(f'plots/prediction_errors_{model_name}.png', dpi=100, bbox_inches='tight')
    plt.close()

In [31]:
# generate feature importance

avg_feature_importance = (
    full_importances
    .dropna(subset=["importance"])
    .groupby(["model", "feature"], as_index=False)
    .agg(
        mean_importance=("importance", "mean"),
        std_importance=("importance", "std")
    )
    .sort_values(["model", "mean_importance"], ascending=[True, False])
)

avg_feature_importance

,model,feature,mean_importance,std_importance
3,Decision Tree,Other Production FC [MWh],0.549140,0.453330
8,Decision Tree,h-1,0.440846,0.448196
88,Decision Tree,h-2,0.001051,0.007242
193,Decision Tree,year,0.001010,0.002047
4,Decision Tree,Photovoltaik Production FC [MWh],0.000815,0.003670
...,...,...,...,...
545,Random Forest,h-82,0.000021,0.000021
402,Random Forest,h-104,0.000021,0.000020
390,Random Forest,Luxembourg_holiday,0.000002,0.000009
389,Random Forest,German_holiday,0.000002,0.000008


In [32]:
# create a plot per model
for model_name in avg_feature_importance["model"].unique():

    temp = (
        avg_feature_importance[avg_feature_importance["model"] == model_name]
        .sort_values("mean_importance", ascending=True)
    )

    plt.figure(figsize=(10, 5))

    plt.barh(
        temp["feature"],
        temp["mean_importance"]
    )

    plt.title(f"Average Feature Importance - {model_name}")
    plt.xlabel("Mean Importance")
    plt.ylabel("Feature")
    plt.grid(axis="x")
    plt.tight_layout()
    plt.savefig(f'plots/feature_importance_{model_name}.png', dpi=100, bbox_inches='tight')
    plt.close()

In [33]:
# pivot: rows = hours, columns = models
mae_by_hour_model = (
    model_performance_real
    .pivot(index="hour", columns="model", values="MAE")
    .sort_index()
)

# grouped bar chart
ax = mae_by_hour_model.plot(
    kind="bar",
    figsize=(18, 6),
    width=0.85
)

plt.title("MAE per Hour and Model", fontsize=16, pad=12)
plt.xlabel("Hour of Day")
plt.ylabel("MAE [€/MWh]")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig("plots/mae_per_hour_grouped_by_model.png", dpi=100, bbox_inches="tight")
plt.close()

print("Grouped MAE plot saved → plots/mae_per_hour_grouped_by_model.png")

Grouped MAE plot saved → plots/mae_per_hour_grouped_by_model.png


In [37]:
# aggregate hourly model performance to overall model performance
model_performance_overall = (
    model_performance_real
    .dropna(subset=["MAE", "RMSE", "R2", "Observations"])
    .assign(
        weighted_MAE=lambda x: x["MAE"] * x["Observations"],
        weighted_RMSE_sq=lambda x: (x["RMSE"] ** 2) * x["Observations"],
        weighted_R2=lambda x: x["R2"] * x["Observations"]
    )
    .groupby("model", as_index=False)
    .agg(
        MAE_sum=("weighted_MAE", "sum"),
        RMSE_sq_sum=("weighted_RMSE_sq", "sum"),
        R2_sum=("weighted_R2", "sum"),
        Observations=("Observations", "sum")
    )
)

# calculate weighted overall metrics
model_performance_overall["MAE"] = (
    model_performance_overall["MAE_sum"] / model_performance_overall["Observations"]
)

model_performance_overall["RMSE"] = np.sqrt(
    model_performance_overall["RMSE_sq_sum"] / model_performance_overall["Observations"]
)

model_performance_overall["R2_weighted_mean"] = (
    model_performance_overall["R2_sum"] / model_performance_overall["Observations"]
)

# keep relevant columns
model_performance_overall = (
    model_performance_overall[
        ["model", "MAE", "RMSE", "R2_weighted_mean", "Observations"]
    ]
    .sort_values("MAE")
    .reset_index(drop=True)
)



In [38]:
# plot overall MAE per model
plt.figure(figsize=(10, 5))

bars = plt.bar(
    model_performance_overall["model"],
    model_performance_overall["MAE"]
)

# add labels
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.2f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.title("Overall MAE per Model", fontsize=16, pad=12)
plt.xlabel("Model")
plt.ylabel("MAE [€/MWh]")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("plots/overall_mae_per_model.png", dpi=100, bbox_inches="tight")
plt.close()

print("Overall MAE plot saved → plots/overall_mae_per_model.png")

Overall MAE plot saved → plots/overall_mae_per_model.png


In [43]:
# create new dataframe for actual vs. predicted plot
preds_scatter = pd.DataFrame(full_predictions).copy()

# keep only valid real-scale predictions
preds_scatter = preds_scatter.dropna(subset=["y_true", "y_pred"])

# actual vs predicted scatter plot per model
for model_name in preds_scatter["model"].unique():

    temp = preds_scatter[
        preds_scatter["model"] == model_name
    ].copy()

    min_val = min(temp["y_true"].min(), temp["y_pred"].min())
    max_val = max(temp["y_true"].max(), temp["y_pred"].max())

    plt.figure(figsize=(7, 7))

    plt.scatter(
        temp["y_true"],
        temp["y_pred"],
        alpha=0.35,
        s=12
    )

    # perfect prediction line
    plt.plot(
        [min_val, max_val],
        [min_val, max_val],
        linestyle="--",
        linewidth=2,
        label="Perfect prediction"
    )

    plt.title(f"Actual vs. Predicted – {model_name}", fontsize=16, pad=12)
    plt.xlabel("Actual Day-Ahead Price [€/MWh]")
    plt.ylabel("Predicted Day-Ahead Price [€/MWh]")
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()

    safe_model_name = (
        model_name
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
    )

    plt.savefig(
        f"plots/actual_vs_predicted_{safe_model_name}.png",
        dpi=100,
        bbox_inches="tight"
    )
    plt.close()

    print(f"Actual vs. Predicted plot saved → plots/actual_vs_predicted_{safe_model_name}.png")

Actual vs. Predicted plot saved → plots/actual_vs_predicted_Decision_Tree.png
Actual vs. Predicted plot saved → plots/actual_vs_predicted_ARX.png
Actual vs. Predicted plot saved → plots/actual_vs_predicted_Random_Forest.png
Actual vs. Predicted plot saved → plots/actual_vs_predicted_Neural_Network.png
Actual vs. Predicted plot saved → plots/actual_vs_predicted_Naive.png


**[TEMPLATE] Interpret your supervised learning results:**

- **Why this model?** *Justify your model choice for your specific prediction task and data type.*
- **Key metric:** *Report and interpret your chosen metric — why is it appropriate for this task?*
- **Baseline comparison:** *How does your best model compare to the simpler baseline?*
- **Limitations:** *Any overfitting concerns? Class imbalance? Feature leakage risks?*

---
## Section 4 — Unsupervised / Generative Block

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] Method choice is justified relative to the structure of the data or task
- [ ] Implementation is correct (k selection, linkage choice, latent dim, …)
- [ ] Output is evaluated with an appropriate measure (silhouette, reconstruction loss, …)
- [ ] Findings are visualised and interpreted in domain terms

---
*This example clusters workers into latent types using K-Means + PCA.
Replace with your method: hierarchical clustering, VAE, GAN, topic model, etc.*

### k-Means Clustering

In [81]:
# reduce the columns which should be used for clustering
feature_cols = [
    'Wind Offshore Production FC [MWh]',
    'Wind Onshore Production FC [MWh]',
    'Photovoltaik Production FC [MWh]',
    'Other Production FC [MWh]',
    "Total Load FC [MWh]",
]

# build the corresponding data frame
df_clustering = df[feature_cols].copy()

# scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clustering)

# calculate the elbow method
inertias = []
cluster_range = range(1, 11)
for k in cluster_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# plot the elbow
plt.figure(figsize=(7, 4))
plt.plot(cluster_range, inertias, marker="o")
plt.xlabel("#Cluster")
plt.ylabel("Inertia")
plt.title("Elbow-Method for K-Means")
plt.grid(True)
plt.savefig("plots/elbow_method.png", dpi=100, bbox_inches="tight")
plt.close()
print('Elbow method plot saved \u2192 plots/elbow_method.png')

Elbow method plot saved → plots/elbow_method.png


In [82]:
# set cluster amount
n_clusters = 4

# Apply K-Means
kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

# add cluster column to data frame to perform cluster analysis
df["cluster"] = kmeans.fit_predict(X_scaled)

# Use t-sne to visualize clusters
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

X_tsne = tsne.fit_transform(X_scaled)

# save results of t-sne in data frame
df["tsne_1"] = X_tsne[:, 0]
df["tsne_2"] = X_tsne[:, 1]

# visualize
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    df["tsne_1"],
    df["tsne_2"],
    c=df["cluster"],
    cmap="tab10",
    alpha=0.7,
    s=30
)

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("K-Means Cluster visualized by t-SNE")

cbar = plt.colorbar(scatter)
cbar.set_label("Cluster")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("plots/tsne_clusters.png", dpi=100, bbox_inches="tight")
plt.close()
print('t-SNE plot saved \u2192 plots/tsne_clusters.png')

t-SNE plot saved → plots/tsne_clusters.png


### Cluster Analysis

In [52]:
## Silhouette Score calculation
sil_score = silhouette_score(X_scaled, df["cluster"])
print(f"Silhouette Score: {sil_score:.4f}")

KeyError: 'cluster'

In [31]:
# Get Boxplots for all features
cols = ["Wind Offshore Production FC [MWh]",
        "Wind Onshore Production FC [MWh]",
        "Photovoltaik Production FC [MWh]",
        "weekday"]
for feature in cols:
    data = [
        df.loc[df["cluster"] == cluster, feature].dropna()for cluster in range(4)]

    plt.figure(figsize=(8, 5))
    plt.boxplot(data, tick_labels=range(4))

    plt.xlabel("Cluster")
    plt.ylabel(feature)
    plt.title(f"Distribution of {feature} by Cluster")
    plt.grid(True)
    plt.savefig(f"plots/clusters_boxplot_{feature}.png", dpi=100, bbox_inches="tight")
    plt.close()
    print(f'{feature} Boxplot saved \u2192 plots/clusters_boxplot_{feature}/.png')

Wind Offshore Production FC [MWh] Boxplot saved → plots/clusters_boxplot_Wind Offshore Production FC [MWh]/.png
Wind Onshore Production FC [MWh] Boxplot saved → plots/clusters_boxplot_Wind Onshore Production FC [MWh]/.png
Photovoltaik Production FC [MWh] Boxplot saved → plots/clusters_boxplot_Photovoltaik Production FC [MWh]/.png
weekday Boxplot saved → plots/clusters_boxplot_weekday/.png


In [32]:
# get histograms per feature and cluster
for feature in cols:
    plt.figure(figsize=(8, 5))

    for cluster in range(4):
        values = df.loc[df["cluster"] == cluster, feature].dropna()

        plt.hist(
            values,
            bins=30,
            alpha=0.5,
            label=f"Cluster {cluster}"
        )

    plt.xlabel(feature)
    plt.ylabel("Quantity")
    plt.title(f"Histogram of {feature} by Cluster")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"plots/clusters_hist_{feature}.png", dpi=100, bbox_inches="tight")
    plt.close()
    print(f'{feature} Histogram saved \u2192 plots/clusters_hist_{feature}/.png')

Wind Offshore Production FC [MWh] Histogram saved → plots/clusters_hist_Wind Offshore Production FC [MWh]/.png
Wind Onshore Production FC [MWh] Histogram saved → plots/clusters_hist_Wind Onshore Production FC [MWh]/.png
Photovoltaik Production FC [MWh] Histogram saved → plots/clusters_hist_Photovoltaik Production FC [MWh]/.png
weekday Histogram saved → plots/clusters_hist_weekday/.png


In [60]:
# get stacked bar charts per feature and cluster
n_bins = 5

for feature in cols: #TODO
    
    temp = df[[feature, "cluster"]].dropna().copy()
    
    # get quantiles
    temp["bin"] = pd.qcut(
        temp[feature],
        q=n_bins,
        duplicates="drop"
    )
    
    # determine cluster percentages
    plot_data = pd.crosstab(
        temp["cluster"],
        temp["bin"],
        normalize="index"
    ) * 100
    
    # stacked bar chart
    plot_data.plot(
        kind="bar",
        stacked=True,
        figsize=(10, 5)
    )
    
    plt.xlabel("Cluster")
    plt.ylabel("Share in %")
    plt.title(f"Distribution of {feature} per Cluster")
    plt.legend(title=feature, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(f"plots/clusters_barchart_{feature}.png", dpi=100, bbox_inches="tight")
    plt.close()
    print(f'{feature} Bar Chart saved \u2192 plots/clusters_barchart_{feature}/.png')

KeyError: "['cluster'] not in index"

Take-Away


**[TEMPLATE] Interpret your unsupervised results:**

- **Why this method?** *Justify in relation to the data structure and research question.*
- **How was k (or another hyperparameter) chosen?** *Describe the trade-off you observed.*
- **What do the clusters mean economically?** *Describe each cluster in plain language: who are these workers?*
- **Limitations:** *Sensitivity to initialisation? Non-spherical clusters? Curse of dimensionality?*

---
## Section 5 — Synthesis & Communication

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] The three method blocks are connected — each result informs the next
- [ ] Conclusions directly answer the research question
- [ ] Limitations and potential confounders are honestly discussed
- [ ] Notebook is readable: clear markdown narrative, labelled plots, no dead code

---
*Replace the toy narrative below with your own synthesis.*

### What causal inference revealed

*[TEMPLATE] Summarise your ATE estimate and what it implies for the research question.
Was the effect statistically and economically meaningful?*

**Toy example:** Backdoor adjustment estimates an ATE of approximately +0.40 log-wage units,
suggesting the training programme substantially raises wages after controlling for age and
education. The random-common-cause refuter confirms the estimate is robust to an added
spurious confounder.

---

### What supervised learning revealed

*[TEMPLATE] What does the predictive model tell you about who participates?
Which features matter most? How does this connect to the causal story?*

**Toy example:** Distance to the training centre is the strongest predictor of participation
(as designed — it is the instrument). The Random Forest outperforms logistic regression
(AUC \u2248 0.85 vs \u2248 0.78), suggesting nonlinear selection effects.

---

### What clustering revealed

*[TEMPLATE] How do the clusters relate to your treatment and outcome?
Do certain worker types benefit more from training?*

**Toy example:** Three worker types emerge: (0) young/low-education workers with the highest
training rate; (1) prime-age/highly-educated workers with the highest baseline wage;
(2) older/medium-education workers with the lowest training rate.

---

### Limitations & honest discussion

*[TEMPLATE — required for full marks] Discuss what your analysis cannot establish.
Examples: unmeasured confounders, external validity, distributional assumptions,
model misspecification, data representativeness.*

---

### Conclusion

*[TEMPLATE] 2–3 sentences directly answering the research question stated in Section 1.*

---
## References

*List all sources in APA format. Include data sources, key papers, and code libraries. Replace the examples below.*

Sharma, A., & Kiciman, E. (2020). DoWhy: An end-to-end library for causal inference. *arXiv:2011.04216*.

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*, 2825\u20132830.

Author, A. A., & Author, B. B. (Year). Title of article. *Journal Name*, *volume*(issue), pages. https://doi.org/...

Dataset: [Name of dataset]. Retrieved from [URL]. Accessed [date].